<a href="https://colab.research.google.com/github/OkyereBiew/bayesian-credit-risk-simulator/blob/main/notebooks/03_bayesian_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
column_names = [
    "checking_account",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_duration",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "target"
]

In [ ]:
df = pd.read_csv(
    "german.data",
    sep=" ",
    header=None,
    names=column_names
)

df.head()

In [ ]:
df["target"] = df["target"].map({
    1: 0,
    2: 1
})

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns

label_encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

In [ ]:
X = df.drop("target", axis=1)

y = df["target"]

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
with pm.Model() as bayesian_logistic_model:

    # Priors for coefficients
    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=X_scaled.shape[1]
    )

    # Prior for intercept
    alpha = pm.Normal(
        "alpha",
        mu=0,
        sigma=1
    )

    # Linear combination
    mu = alpha + pm.math.dot(X_scaled, beta)

    # Logistic transformation
    theta = pm.Deterministic(
        "theta",
        pm.math.sigmoid(mu)
    )

    # Likelihood
    likelihood = pm.Bernoulli(
        "likelihood",
        p=theta,
        observed=y
    )

In [ ]:
with bayesian_logistic_model:

    trace = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        random_seed=42
    )

In [ ]:
az.summary(trace)

In [ ]:
az.plot_posterior(
    trace,
    var_names=["alpha"]
)

plt.show()

In [ ]:
az.plot_trace(
    trace,
    var_names=["alpha"]
)

plt.show()

## Bayesian Modeling Interpretation

The Bayesian logistic regression model estimates the probability of credit default while explicitly accounting for uncertainty in parameter estimation.

Unlike traditional logistic regression, Bayesian inference provides posterior distributions for model parameters, allowing probabilistic interpretation of financial risk.

The posterior plots and trace diagnostics help evaluate uncertainty, parameter stability, and convergence behavior during MCMC sampling.